# Stage 2 Experiment Runner

Use this notebook to run the next controlled post-training experiments on top of the reduced-schema QLoRA baseline.

Recommended preset order:

- `data_small_600`
- `curriculum_simple_medium_then_full`
- `rank8_full`
- `rank16_full`
- `rank32_full`

Each preset trains, exports test predictions, evaluates raw predictions, applies repair, and writes repaired reports.

In [1]:
# Uncomment this once in a fresh online environment if needed.
# %pip install -q transformers datasets peft accelerate trl bitsandbytes pyyaml jsonschema ipywidgets

In [1]:
!nvidia-smi

Sun Mar 22 04:46:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200 NVL                On  |   00000000:00:05.0 Off |                    0 |
| N/A   26C    P0             71W /  600W |       4MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import os
import random
import sys

import torch


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('project_root =', PROJECT_ROOT)
print('python =', sys.version)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())

from datasets import load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

from src.data.io import dump_jsonl, load_jsonl
from src.evaluation.field_analysis import analyze_field_errors
from src.evaluation.metrics import evaluate_sample, summarize_results, try_parse_prediction_text
from src.evaluation.reporting import group_sample_results, write_json_report
from src.inference.repair import repair_prediction
from src.schemas.registry import get_schema
from src.training.formatters import DEFAULT_SYSTEM_PROMPT, build_user_prompt, convert_to_sft_records

project_root = /home/lyan11/small-llm-structured-posttraining
python = 3.13.11 | packaged by conda-forge | (main, Dec  6 2025, 11:24:03) [GCC 14.3.0]
cuda_available = True
device = NVIDIA H200 NVL
bf16_supported = True


In [4]:
PRESETS = {
    'data_small_600': {
        'experiment_name': 'qwen25_3b_stage2_data_small_600',
        'mode': 'subset',
        'subset_size': 600,
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 3,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'curriculum_simple_medium_then_full': {
        'experiment_name': 'qwen25_3b_stage2_curriculum_sm_then_full',
        'mode': 'curriculum',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'stage1_epochs': 1,
        'stage2_epochs': 2,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
        'stage1_buckets': ['simple', 'medium'],
    },
    'rank8_full': {
        'experiment_name': 'qwen25_3b_stage2_rank8_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 8,
        'alpha': 16,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 3,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 3,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'rank32_full': {
        'experiment_name': 'qwen25_3b_stage2_rank32_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 32,
        'alpha': 64,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 3,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'epoch2_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_epoch2_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 2,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'epoch3_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_epoch3_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 3,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'epoch5_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_epoch5_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 5,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'epoch7_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_epoch7_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 7,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'epoch9_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_epoch9_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 9,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
}

SELECTED_PRESET = 'epoch9_rank16_full'
config = PRESETS[SELECTED_PRESET]
config

{'experiment_name': 'qwen25_3b_stage2_epoch9_rank16_full',
 'mode': 'full',
 'base_model': 'Qwen/Qwen2.5-3B-Instruct',
 'rank': 16,
 'alpha': 32,
 'dropout': 0.05,
 'learning_rate': 0.0002,
 'epochs': 9,
 'batch_size': 8,
 'grad_accum': 4,
 'max_seq_length': 2048,
 'seed': 42,
 'schema_name': 'ticket_schema_v1_reduced',
 'include_schema_definition': False}

In [5]:
ARTIFACT_DIR = PROJECT_ROOT / 'data' / 'stage2'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

train_records = load_jsonl(PROJECT_ROOT / 'data' / 'reduced' / 'phase1_train_reduced.jsonl')
val_records = load_jsonl(PROJECT_ROOT / 'data' / 'reduced' / 'phase1_val_reduced.jsonl')
test_records = load_jsonl(PROJECT_ROOT / 'data' / 'reduced' / 'phase1_test_reduced.jsonl')

print('train =', len(train_records))
print('val =', len(val_records))
print('test =', len(test_records))


def bucket_counts(records):
    counts = defaultdict(int)
    for record in records:
        counts[record['complexity_bucket']] += 1
    return dict(sorted(counts.items()))


def stratified_subset(records, total_size, seed):
    grouped = defaultdict(list)
    for record in records:
        grouped[record['complexity_bucket']].append(record)
    rng = random.Random(seed)
    selected = []
    total_records = len(records)
    for bucket_name, bucket_records in grouped.items():
        rng.shuffle(bucket_records)
        ratio = len(bucket_records) / total_records
        bucket_take = max(1, round(total_size * ratio))
        selected.extend(bucket_records[:bucket_take])
    rng.shuffle(selected)
    if len(selected) > total_size:
        selected = selected[:total_size]
    while len(selected) < total_size:
        selected.append(records[len(selected) % len(records)])
    return selected


def select_train_records(records, preset):
    mode = preset['mode']
    if mode == 'full':
        return records
    if mode == 'subset':
        return stratified_subset(records, total_size=preset['subset_size'], seed=preset['seed'])
    if mode == 'curriculum':
        return records
    raise ValueError(f'Unsupported mode: {mode}')


selected_train_records = select_train_records(train_records, config)
print('selected_train =', len(selected_train_records))
print('selected_train_buckets =', bucket_counts(selected_train_records))

train = 1993
val = 247
test = 254
selected_train = 1993
selected_train_buckets = {'complex': 132, 'medium': 1392, 'simple': 469}


In [6]:
experiment_name = config['experiment_name']


def write_sft_split(records, file_name, include_schema_definition=False):
    path = ARTIFACT_DIR / file_name
    dump_jsonl(path, convert_to_sft_records(records, include_schema_definition=include_schema_definition))
    return path


if config['mode'] == 'curriculum':
    stage1_records = [
        record
        for record in train_records
        if record['complexity_bucket'] in set(config['stage1_buckets'])
    ]
    stage2_records = list(train_records)
    stage1_train_path = write_sft_split(
        stage1_records,
        f"{experiment_name}_stage1_train.jsonl",
        include_schema_definition=config['include_schema_definition'],
    )
    stage2_train_path = write_sft_split(
        stage2_records,
        f"{experiment_name}_stage2_train.jsonl",
        include_schema_definition=config['include_schema_definition'],
    )
else:
    train_path = write_sft_split(
        selected_train_records,
        f"{experiment_name}_train.jsonl",
        include_schema_definition=config['include_schema_definition'],
    )

val_path = write_sft_split(
    val_records,
    f"{experiment_name}_val.jsonl",
    include_schema_definition=config['include_schema_definition'],
)

print('val_path =', val_path)
if config['mode'] == 'curriculum':
    print('stage1_train_path =', stage1_train_path)
    print('stage2_train_path =', stage2_train_path)
    print('stage1_buckets =', bucket_counts(stage1_records))
else:
    print('train_path =', train_path)

val_path = /home/lyan11/small-llm-structured-posttraining/data/stage2/qwen25_3b_stage2_epoch9_rank16_full_val.jsonl
train_path = /home/lyan11/small-llm-structured-posttraining/data/stage2/qwen25_3b_stage2_epoch9_rank16_full_train.jsonl


In [7]:
base_model = config['base_model']
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


def load_chat_dataset(train_file, validation_file):
    dataset = load_dataset(
        'json',
        data_files={
            'train': str(train_file),
            'validation': str(validation_file),
        },
    )

    def format_chat_example(example):
        example['text'] = tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
        return example

    return dataset.map(format_chat_example)


print('pad_token =', tokenizer.pad_token)
print('eos_token =', tokenizer.eos_token)

pad_token = <|endoftext|>
eos_token = <|im_end|>


In [8]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)


def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False
    return model


def build_training_args(output_dir, num_epochs):
    return TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=float(config['learning_rate']),
        num_train_epochs=float(num_epochs),
        per_device_train_batch_size=int(config['batch_size']),
        per_device_eval_batch_size=int(config['batch_size']),
        gradient_accumulation_steps=int(config['grad_accum']),
        warmup_steps=50,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        bf16=use_bf16,
        fp16=not use_bf16,
        report_to='none',
        remove_unused_columns=False,
        seed=int(config['seed']),
    )


def build_peft_config():
    return LoraConfig(
        r=int(config['rank']),
        lora_alpha=int(config['alpha']),
        lora_dropout=float(config['dropout']),
        bias='none',
        task_type=TaskType.CAUSAL_LM,
        target_modules='all-linear',
    )


def build_trainer(model, dataset, output_dir, num_epochs, peft_config=None):
    training_args = build_training_args(output_dir=output_dir, num_epochs=num_epochs)
    trainer_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': dataset['train'],
        'eval_dataset': dataset['validation'],
        'processing_class': tokenizer,
    }
    if peft_config is not None:
        trainer_kwargs['peft_config'] = peft_config
    return SFTTrainer(**trainer_kwargs)

In [9]:
output_root = PROJECT_ROOT / 'results' / 'checkpoints' / experiment_name
model = load_base_model()

if config['mode'] == 'curriculum':
    stage1_dataset = load_chat_dataset(stage1_train_path, val_path)
    stage1_output = output_root / 'stage1'
    stage1_trainer = build_trainer(
        model=model,
        dataset=stage1_dataset,
        output_dir=stage1_output,
        num_epochs=config['stage1_epochs'],
        peft_config=build_peft_config(),
    )
    stage1_result = stage1_trainer.train()
    stage1_trainer.save_model(str(stage1_output))
    model = stage1_trainer.model
    print('stage1_train_loss =', stage1_result.training_loss)
    print('stage1 model updated with adapters and ready for continuation')

    stage2_dataset = load_chat_dataset(stage2_train_path, val_path)
    stage2_output = output_root / 'stage2'
    stage2_trainer = build_trainer(
        model=model,
        dataset=stage2_dataset,
        output_dir=stage2_output,
        num_epochs=config['stage2_epochs'],
        peft_config=None,
    )
    stage2_result = stage2_trainer.train()
    stage2_trainer.save_model(str(output_root))
    print('stage2_train_loss =', stage2_result.training_loss)
else:
    dataset = load_chat_dataset(train_path, val_path)
    trainer = build_trainer(
        model=model,
        dataset=dataset,
        output_dir=output_root,
        num_epochs=config['epochs'],
        peft_config=build_peft_config(),
    )
    train_result = trainer.train()
    trainer.save_model(str(output_root))
    print('train_loss =', train_result.training_loss)

print('saved_checkpoint_dir =', output_root)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1993 [00:00<?, ? examples/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/247 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/247 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.724612,0.681373
100,0.534103,0.520496
150,0.424012,0.438064
200,0.371032,0.398140
250,0.343352,0.374748
300,0.292201,0.370557
350,0.278545,0.365297
400,0.240320,0.376345
450,0.224461,0.387014
500,0.204817,0.383229


train_loss = 0.4649940809034586
saved_checkpoint_dir = /home/lyan11/small-llm-structured-posttraining/results/checkpoints/qwen25_3b_stage2_epoch9_rank16_full


In [10]:
generation_kwargs = {
    'max_new_tokens': 256,
    'do_sample': False,
    'temperature': 1.0,
    'top_p': 1.0,
}


def build_inference_messages(record):
    return [
        {
            'role': 'system',
            'content': DEFAULT_SYSTEM_PROMPT,
        },
        {
            'role': 'user',
            'content': build_user_prompt(
                input_text=record['input_text'],
                task_name=record['task_name'],
                schema_name=record['schema_name'],
                include_schema_definition=config['include_schema_definition'],
            ),
        },
    ]


def generate_prediction_text(record):
    messages = build_inference_messages(record)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, **generation_kwargs)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


predictions = []
for idx, record in enumerate(test_records):
    prediction_text = generate_prediction_text(record)
    try:
        prediction_json = json.loads(prediction_text)
    except json.JSONDecodeError:
        prediction_json = None
    predictions.append(
        {
            'sample_id': record['sample_id'],
            'prediction_text': prediction_text,
            'prediction_json': prediction_json,
            'metadata': {
                'model_name': base_model,
                'experiment_id': experiment_name,
            },
        }
    )
    if (idx + 1) % 25 == 0:
        print(f'generated {idx + 1} / {len(test_records)}')

prediction_path = PROJECT_ROOT / 'results' / 'predictions' / f'{experiment_name}_test.jsonl'
dump_jsonl(prediction_path, predictions)
print('prediction_path =', prediction_path)
print('num_predictions =', len(predictions))

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generated 25 / 254
generated 50 / 254
generated 75 / 254
generated 100 / 254
generated 125 / 254
generated 150 / 254
generated 175 / 254
generated 200 / 254
generated 225 / 254
generated 250 / 254
prediction_path = /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_stage2_epoch9_rank16_full_test.jsonl
num_predictions = 254


In [11]:
schema = get_schema(config['schema_name'])


def sample_eval_dicts(gold_records, pred_records):
    predictions_by_id = {record['sample_id']: record for record in pred_records}
    results = []
    for gold_record in gold_records:
        pred_record = predictions_by_id.get(gold_record['sample_id'], {})
        sample_eval = evaluate_sample(
            sample_id=gold_record['sample_id'],
            prediction_text=pred_record.get('prediction_text'),
            prediction_json=pred_record.get('prediction_json'),
            target_json=gold_record['target_json'],
            schema=schema,
        )
        results.append(
            {
                **sample_eval.__dict__,
                'schema_name': gold_record['schema_name'],
                'complexity_bucket': gold_record.get('complexity_bucket', 'unknown'),
            }
        )
    return results


def summarize_from_dicts(sample_results):
    from src.evaluation.metrics import SampleEvaluation

    evaluations = [
        SampleEvaluation(
            sample_id=item['sample_id'],
            valid_json=item['valid_json'],
            schema_compliant=item['schema_compliant'],
            field_exact_match=item['field_exact_match'],
            exact_match=item['exact_match'],
            primary_error=item['primary_error'],
        )
        for item in sample_results
    ]
    return summarize_results(evaluations)


raw_sample_results = sample_eval_dicts(test_records, predictions)
raw_report = {
    'summary': summarize_from_dicts(raw_sample_results),
    'grouped_summary': {
        'by_complexity_bucket': {
            name: summarize_from_dicts(items)
            for name, items in group_sample_results(raw_sample_results, 'complexity_bucket').items()
        },
    },
    'per_sample': raw_sample_results,
}

raw_report_path = PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_test_report.json'
raw_field_path = PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_field_analysis.json'
write_json_report(raw_report_path, raw_report)
write_json_report(raw_field_path, analyze_field_errors(test_records, predictions))

repaired_predictions = []
for record in predictions:
    prediction_json = record.get('prediction_json')
    prediction_text = record.get('prediction_text')
    if prediction_json is None and isinstance(prediction_text, str):
        _, prediction_json = try_parse_prediction_text(prediction_text)
    repaired_json, repaired = repair_prediction(prediction_json, schema)
    repaired_predictions.append(
        {
            **record,
            'prediction_json': repaired_json,
            'metadata': {
                **record.get('metadata', {}),
                'repaired': repaired,
            },
        }
    )

repaired_sample_results = sample_eval_dicts(test_records, repaired_predictions)
repaired_report = {
    'summary': summarize_from_dicts(repaired_sample_results),
    'per_sample': repaired_sample_results,
}

repaired_pred_path = PROJECT_ROOT / 'results' / 'predictions' / f'{experiment_name}_test_repaired.jsonl'
repaired_report_path = PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_test_repaired_report.json'
repaired_field_path = PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_test_repaired_field_analysis.json'

dump_jsonl(repaired_pred_path, repaired_predictions)
write_json_report(repaired_report_path, repaired_report)
write_json_report(repaired_field_path, analyze_field_errors(test_records, repaired_predictions))

print('raw_report =', raw_report_path)
print('raw_field =', raw_field_path)
print('repaired_pred =', repaired_pred_path)
print('repaired_report =', repaired_report_path)
print('repaired_field =', repaired_field_path)

raw_report = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_full_test_report.json
raw_field = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_full_field_analysis.json
repaired_pred = /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_stage2_epoch9_rank16_full_test_repaired.jsonl
repaired_report = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_full_test_repaired_report.json
repaired_field = /home/lyan11/small-llm-structured-posttraining/results/metrics/qwen25_3b_stage2_epoch9_rank16_full_test_repaired_field_analysis.json


In [12]:
print('raw_summary =')
print(json.dumps(raw_report['summary'], indent=2))
print()
print('repaired_summary =')
print(json.dumps(repaired_report['summary'], indent=2))

raw_summary =
{
  "num_samples": 254,
  "valid_json_rate": 1.0,
  "schema_compliance_rate": 1.0,
  "field_exact_match": 0.9166070150322119,
  "end_to_end_exact_match": 0.5708661417322834,
  "error_counts": {
    "invalid_json": 0,
    "schema_violation": 0,
    "missing_required_field": 0,
    "enum_error": 0,
    "type_error": 0,
    "value_hallucination": 109,
    "extraction_omission": 0,
    "cross_field_inconsistency": 0
  }
}

repaired_summary =
{
  "num_samples": 254,
  "valid_json_rate": 1.0,
  "schema_compliance_rate": 1.0,
  "field_exact_match": 0.9166070150322119,
  "end_to_end_exact_match": 0.5708661417322834,
  "error_counts": {
    "invalid_json": 0,
    "schema_violation": 0,
    "missing_required_field": 0,
    "enum_error": 0,
    "type_error": 0,
    "value_hallucination": 109,
    "extraction_omission": 0,
    "cross_field_inconsistency": 0
  }
}
